In [2]:
import hashlib

def calculate_sha256(filepath):
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        # อ่านทีละ 4096 bytes แทนการโหลดทั้งไฟล์เข้า memory
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

files = ["secom.data", "secom_labels.data", "secom.names"]

registry = {}
for fname in files:
    file_hash = calculate_sha256(f"secom/{fname}")
    registry[fname] = file_hash
    print(f"{fname}: {file_hash}")

secom.data: 20f0e7ee434f7dcbae0eea9ffff009a2b57f42d6b0dc9a5bd4f00782c0a3374c
secom_labels.data: 126884cf453705c9e61a903fe906f0665a3b45ce3639e621edc5c93c89627e03
secom.names: 6d91b0b46cdee03064ee3e3112f937c1b3f7fcd9933575794ec07974e6f1ea59


01_secom_data_audit.py

In [3]:
import json
import os
from datetime import datetime

registry_path = "data_registry.json"

# ลบไฟล์เก่าถ้ามีอยู่แล้ว
if os.path.exists(registry_path):
    os.remove(registry_path)
    print(f"ลบไฟล์เก่า {registry_path} เรียบร้อย")

registry_record = {
    "dataset": "SECOM",
    "source": "UCI Machine Learning Repository (ID: 179)",
    "download_date": datetime.now().strftime("%Y-%m-%d"),
    "files": registry
}

with open(registry_path, "w") as f:
    json.dump(registry_record, f, indent=2)

print(f"สร้างไฟล์ {registry_path} ใหม่เรียบร้อย")
print(json.dumps(registry_record, indent=2))

ลบไฟล์เก่า data_registry.json เรียบร้อย
สร้างไฟล์ data_registry.json ใหม่เรียบร้อย
{
  "dataset": "SECOM",
  "source": "UCI Machine Learning Repository (ID: 179)",
  "download_date": "2026-09-21",
  "files": {
    "secom.data": "20f0e7ee434f7dcbae0eea9ffff009a2b57f42d6b0dc9a5bd4f00782c0a3374c",
    "secom_labels.data": "126884cf453705c9e61a903fe906f0665a3b45ce3639e621edc5c93c89627e03",
    "secom.names": "6d91b0b46cdee03064ee3e3112f937c1b3f7fcd9933575794ec07974e6f1ea59"
  }
}


In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# secom.data: whitespace-separated, ไม่มี header
X = pd.read_csv("secom/secom.data", sep=r"\s+", header=None)

# secom_labels.data: column 1 = label (+1/-1), column 2 = timestamp
y = pd.read_csv("secom/secom_labels.data", sep=r"\s+", header=None,
                           names=["label", "timestamp"])

print("X.shape:", X.shape)
print("y.shape:", y.shape)
print("the number of predictor variables:", X.shape[1])

X.shape: (1567, 590)
y.shape: (1567, 2)
the number of predictor variables: 590


In [5]:
display(y.head())

,label,timestamp
0,-1,19/07/2008 11:55:00
1,-1,19/07/2008 12:32:00
2,1,19/07/2008 13:17:00
3,-1,19/07/2008 14:43:00
4,-1,19/07/2008 15:22:00


In [6]:
display(X.head())

,0,1,2,3,4,5,6,7,8,9,...,580,581,582,583,584,585,586,587,588,589
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.0044,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,NaN,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432


In [ ]:

def missing_rate_filter(X, threshold):
    miss_rate = pd.DataFrame(X.isna().sum() / len(X) * 100).reset_index()
    miss_rate.columns = ["feature", "missing_rate"]
    miss_by_thres = miss_rate[miss_rate["missing_rate"] > threshold]
    print(len(miss_by_thres), f"features have more than {threshold}% missing values")
    return miss_by_thres["feature"].tolist()


,feature,missing_rate
0,0,0.382897
1,1,0.446713
2,2,0.893427
3,3,0.893427
4,4,0.893427
...,...,...
585,585,0.063816
586,586,0.063816
587,587,0.063816
588,588,0.063816


,feature,missing_rate
72,72,50.670070
73,73,50.670070
85,85,85.577537
109,109,64.964901
110,110,64.964901
111,111,64.964901
157,157,91.193363
158,158,91.193363
220,220,85.577537
244,244,64.964901


28 features have more than 50% missing values
[72, 73, 85, 109, 110, 111, 157, 158, 220, 244, 245, 246, 292, 293, 345, 346, 358, 382, 383, 384, 492, 516, 517, 518, 578, 579, 580, 581]


In [23]:
def audit_correlation_structure(X):
    corr_matrix = X.corr().abs()

    upper_mask = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    upper = corr_matrix.where(upper_mask)

    thresholds = [0.90, 0.95, 0.98, 0.995]
    result = {}
    pairs_by_threshold = {}

    all_pairs = upper.stack().dropna().reset_index()
    all_pairs.columns = ["feature_1", "feature_2", "correlation"]

    for t in thresholds:
        count = int((upper >= t).sum().sum())
        result[f">=|{t}|"] = count

        matched_pairs = all_pairs[all_pairs["correlation"] >= t].sort_values("correlation", ascending=False).reset_index(drop=True)
        pairs_by_threshold[f">=|{t}|"] = matched_pairs

    return result, upper, pairs_by_threshold


correlation_summary, upper_matrix, pairs_by_threshold = audit_correlation_structure(X)

print(pairs_by_threshold[">=|0.9|"]["feature_1"].tolist())

[220, 209, 578, 579, 74, 206, 74, 74, 492, 580, 74, 581, 342, 209, 85, 209, 358, 206, 206, 347, 74, 206, 342, 34, 140, 172, 307, 152, 244, 244, 252, 382, 4, 4, 246, 251, 249, 583, 176, 389, 268, 251, 248, 147, 245, 177, 187, 148, 175, 253, 147, 195, 384, 254, 246, 135, 181, 221, 282, 148, 383, 245, 551, 154, 172, 174, 143, 156, 267, 153, 360, 197, 550, 248, 138, 183, 283, 255, 386, 136, 551, 207, 177, 198, 174, 172, 203, 222, 185, 164, 312, 223, 160, 165, 219, 435, 564, 137, 252, 169, 153, 224, 390, 554, 155, 584, 549, 213, 170, 572, 583, 161, 288, 180, 185, 154, 157, 173, 572, 434, 212, 163, 289, 144, 159, 516, 244, 382, 225, 156, 142, 384, 244, 382, 492, 220, 220, 434, 246, 244, 246, 283, 152, 574, 93, 205, 443, 291, 184, 148, 148, 202, 222, 543, 321, 99, 166, 151, 267, 92, 182, 565, 211, 339, 162, 254, 341, 323, 187, 333, 217, 171, 253, 405, 391, 392, 205, 198, 117, 117, 250, 139, 255, 204, 207, 559, 287, 152, 564, 203, 345, 197, 319, 334, 183, 393, 182, 25, 218, 573, 550, 221, 311,

In [241]:

class_distribution = pd.DataFrame(y["label"].value_counts())
class_distribution["percentage"] = round((class_distribution["count"] / class_distribution["count"].sum()) * 100, 2)
class_distribution.reset_index(inplace=True)
class_distribution['label'] = class_distribution['label'].map({-1: 'Pass', 1: 'Fail'})
display(class_distribution)

,label,count,percentage
0,Pass,1463,93.36
1,Fail,104,6.64
